# SEC Filing Fee (Exhibit 107) Explorer

Query SEC EDGAR for **Filing Fee Disclosure** data — the structured XBRL fee tables required in registration statements since February 2022 (Exhibit 107). The code here is similar to the tools in the `filing-fee-mcp` server:

| Function | MCP Tool | Description |
|:---------|:---------|:------------|
| `search_fee_filings()` | `search_fee_filings` | Find registration filings (S-1, S-3, etc.) for a company |
| `get_fee_exhibit()` | `get_fee_exhibit` | Fetch and parse an Exhibit 107 from a specific filing |
| `get_company_fee_history()` | `get_company_fee_history` | Parse all fee exhibits for a company |
| `compare_fee_rates()` | `compare_fee_rates` | Compare fee data across 2-8 ENTITIES |
| `get_ffd_concepts()` | `get_ffd_concepts_by_company` | Get all ffd: XBRL facts (companyfacts API + Exhibit 107 backfill) |
| `search_filings_by_fee()` | `search_filings_by_fee` | Full-text search scoped to registration statements |
| `export_fee_data()` | `export_fee_data` | Export fee data to CSV or combined .xlsx workbook |


In [ ]:
# @title Define variables
# @markdown List entities (comma-separated) by ticker or name, dates to search (YYYY-MM-DD), and any text string to find, then run this cell for tables that include correspoding reports, concepts and facts, and any search results.
ENTITIES = "klarna, tesla, CHYM" # @param {"type":"string","placeholder":"klarna, tesla, CHYM"}
TICKERS_STR = ENTITIES
TICKERS = [t.strip() for t in TICKERS_STR.split(',') if t.strip()]
START_DATE = "" # @param {"type":"string","placeholder":"2023-01-01"}
END_DATE = "" # @param {"type":"string","placeholder":""}
TEXT = "musk" # @param {"type":"string","placeholder":"musk"}
print ("Update the name and email for the SEC API's USER_AGENT (required) below then run the cell \nto initialize the code for use. \n\nThe next cell only needs to be run once per session.")

In [ ]:
# @title Update the name and email detail below for the SEC API.
# @markdown This cell loads functions related to the code in the next cell and **ONLY NEEDS TO BE RUN ONCE PER SESSION**.
# Notebook-level configuration
USER_AGENT = "your-name your-email@example.com" # @param {"type":"string","placeholder":"your-name your-email@example.com"}
if USER_AGENT.strip() == "your-name your-email@example.com" or not USER_AGENT.strip():
    raise ValueError(
        "Please replace USER_AGENT with your name and email before running this cell."
    )

%pip install openpyxl beautifulsoup4
import requests
import pandas as pd
import re
import time
import csv
import io
from typing import Optional
from datetime import date
from datetime import datetime, timedelta
from bs4 import BeautifulSoup


# SEC API base URLs
_BASE = "https://data.sec.gov"
_EDGAR = "https://www.sec.gov"
_EFTS = "https://efts.sec.gov"

# Rate limiter (SEC allows 10 req/s)
_last_request_at = 0.0

def _sec_get(url: str) -> requests.Response:
    global _last_request_at
    elapsed = time.time() - _last_request_at
    if elapsed < 0.1:
        time.sleep(0.1 - elapsed)
    _last_request_at = time.time()

    headers = {"User-Agent": USER_AGENT, "Accept": "application/json, text/html, */*"}
    for attempt in range(3):
        resp = requests.get(url, headers=headers, timeout=30)
        if resp.ok or resp.status_code == 404:
            return resp
        if resp.status_code in (429, 500, 502, 503, 504) and attempt < 2:
            time.sleep(1.0 * (2 ** attempt))
            continue
        resp.raise_for_status()
    raise RuntimeError(f"SEC request failed after retries: {url}")

def _sec_json(url: str):
    resp = _sec_get(url)
    if resp.status_code == 404:
        return None
    return resp.json()

def _sec_text(url: str) -> Optional[str]:
    resp = _sec_get(url)
    if resp.status_code == 404:
        return None
    return resp.text

# Registration form types that carry Exhibit 107
REGISTRATION_FORMS = [
    "S-1", "S-1/A", "S-3", "S-3/A", "S-4", "S-4/A", "S-11", "S-11/A",
    "F-1", "F-1/A", "F-3", "F-3/A", "F-4", "F-4/A", "F-6", "F-6/A",
    "424B1", "424B2", "424B3", "424B4", "424B5", "424B7", "424B8",
    "SC TO-I", "SC TO-T", "PREM14A", "DEFM14A", "POS AM", "S-8", "S-8/A",
]
# CIK resolution (ticker → CIK)

_ticker_cache: dict = {}
_ticker_cache_time: float = 0.0

def _get_ticker_map() -> dict:
    global _ticker_cache, _ticker_cache_time
    if _ticker_cache and (time.time() - _ticker_cache_time < 3600):
        return _ticker_cache
    data = _sec_json(f"{_EDGAR}/files/company_tickers.json")
    _ticker_cache = data or {}
    _ticker_cache_time = time.time()
    return _ticker_cache

def resolve_cik(query: str) -> dict:
    """Resolve a ticker, company name, or CIK to {'cik': str, 'name': str, 'ticker': str|None}."""
    q = query.strip()
    if q.isdigit():
        return {"cik": q.zfill(10), "name": q, "ticker": None}

    tmap = _get_ticker_map()
    upper = q.upper()
    for entry in tmap.values():
        if entry["ticker"].upper() == upper:
            return {
                "cik": str(entry["cik_str"]).zfill(10),
                "name": entry["title"],
                "ticker": entry["ticker"],
            }
    lower = q.lower()
    for entry in tmap.values():
        if lower in entry["title"].lower():
            return {
                "cik": str(entry["cik_str"]).zfill(10),
                "name": entry["title"],
                "ticker": entry["ticker"],
            }
    raise ValueError(f'Could not resolve "{query}" to a CIK. Try a ticker symbol or CIK number.')


def get_company_filings(cik: str) -> dict:
    """Fetch the submissions JSON for a CIK."""
    return _sec_json(f"{_BASE}/submissions/CIK{cik.zfill(10)}.json")


def get_company_facts(cik: str) -> dict:
    """Fetch all XBRL companyfacts for a CIK."""
    return _sec_json(f"{_BASE}/api/xbrl/companyfacts/CIK{cik.zfill(10)}.json")


def _parse_index_headers(html: str) -> list:
    """Parse the SGML index-headers HTML to extract document TYPE and FILENAME.

    Each document block looks like:
      &lt;DOCUMENT&gt;
      &lt;TYPE&gt;EX-FILING FEES
      &lt;SEQUENCE&gt;2
      &lt;FILENAME&gt;exhibit107-fx1a.htm
      &lt;DESCRIPTION&gt;EX-FILING FEES
    """
    docs = []
    for m in re.finditer(r"&lt;DOCUMENT&gt;([\s\S]*?)(?:&lt;/DOCUMENT&gt;|$)", html, re.I):
        block = m.group(1);
        type_m = re.search(r"&lt;TYPE&gt;\s*([^\n<]+)", block, re.I)
        file_m = re.search(r"&lt;FILENAME&gt;\s*([^\n<]+)", block, re.I)
        desc_m = re.search(r"&lt;DESCRIPTION&gt;\s*([^\n<]+)", block, re.I)
        if type_m and file_m:
            doc = {
                "name": file_m.group(1).strip(),
                "type": type_m.group(1).strip(),
                "size": "0",
            }
            if desc_m:
                doc["description"] = desc_m.group(1).strip()
            docs.append(doc)
    return docs


def _parse_index_htm(html: str) -> list:
    """Parse an EDGAR index.htm page and return the listed document names."""
    docs = []
    soup = BeautifulSoup(html, "html.parser")

    for link in soup.find_all("a", href=True):
        href = link["href"].strip()
        if not href:
            continue
        if href.lower().endswith((".htm", ".html")):
            name = href.split("#", 1)[0]
            if name and name not in ("index.htm", "index.html"):
                docs.append({"name": name, "type": None, "size": "0"})

    return docs


def get_filing_documents(cik: str, accession: str) -> list:
    """Fetch the document index for a filing via SGML index-headers or fallback to index.htm.

    Uses index-headers.html when available for TYPE and FILENAME data.
    Falls back to the standard EDGAR index.htm listing to capture all .htm/.html
    documents when index-headers metadata is unavailable.
    """
    acc_no_dash = accession.replace("-", "")
    cik_int = int(cik)
    headers_url = f"{_EDGAR}/Archives/edgar/data/{cik_int}/{acc_no_dash}/{accession}-index-headers.html"
    html = _sec_text(headers_url)
    if html:
        docs = _parse_index_headers(html)
        if docs:
            return docs

    index_url = build_edgar_url(cik, accession)
    html = _sec_text(index_url)
    if html:
        return _parse_index_htm(html)

    return []


def get_filing_document_text(cik: str, accession: str, filename: str) -> Optional[str]:
    """Fetch raw text/HTML of a specific document."""
    acc_no_dash = accession.replace("-", "")
    cik_int = int(cik)
    url = f"{_EDGAR}/Archives/edgar/data/{cik_int}/{acc_no_dash}/{filename}"
    return _sec_text(url)


def build_edgar_url(cik: str, accession: str, filename: str = "") -> str:
    acc_no_dash = accession.replace("-", "")
    cik_int = int(cik)
    if filename:
        return f"{_EDGAR}/Archives/edgar/data/{cik_int}/{acc_no_dash}/{filename}"
    return f"{_EDGAR}/Archives/edgar/data/{cik_int}/{acc_no_dash}/{accession}-index.htm"
# Exhibit 107 iXBRL Parser

_FEE_EXHIBIT_TYPES = {"EX-FILING FEES", "EXFILINGFEES", "107"}

# High-confidence patterns that unambiguously identify fee exhibits.
_FEE_FILENAME_PRIMARY = [
    re.compile(r"exfilingfee", re.I),
    re.compile(r"ex-filing", re.I),
    re.compile(r"filingfee", re.I),
]

# Lower-confidence patterns — "exhibit107" is ambiguous because filers also
# name Exhibit 10.7 as "exhibit107". We try these only after primary patterns
# fail, and validate the parsed result looks like a real fee table.
_FEE_FILENAME_SECONDARY = [
    re.compile(r"ex107", re.I),
    re.compile(r"exhibit107", re.I),
]


def _clean_xbrl_value(raw: str) -> str:
    """Strip HTML tags and decode entities."""
    v = re.sub(r"<[^>]+>", "", raw)
    for entity, char in [("&amp;", "&"), ("&lt;", "<"), ("&gt;", ">"), ("&nbsp;", " ")]:
        v = v.replace(entity, char)
    v = re.sub(r"&#\d+;", "", v)
    return re.sub(r"\s+", " ", v).strip()


def _parse_number(raw: Optional[str]) -> Optional[float]:
    if not raw:
        return None
    cleaned = re.sub(r"[$,\s]", "", raw)
    try:
        return float(cleaned)
    except ValueError:
        return None


def _parse_security_type(raw: str) -> str:
    u = raw.lower()
    if any(kw in u for kw in ("debt", "note", "bond", "debenture")):
        return "Debt"
    if any(kw in u for kw in ("equit", "common", "preferred", "warrant", "unit")):
        return "Equity"
    return "Other" if raw else "Unknown"


def _extract_xbrl_facts(html: str) -> list:
    """Extract ffd: XBRL facts from iXBRL HTML."""
    facts = []

    # iXBRL inline facts
    ix_pat = re.compile(
        r'<ix:(nonNumeric|nonFraction)[^>]*?name="ffd:([^"]+)"[^>]*?contextRef="([^"]*)"'
        r'[^>]*?(?:decimals="([^"]*)")?[^>]*?(?:unitRef="([^"]*)")?[^>]*?>'
        r'([\s\S]*?)</ix:\1>',
        re.I,
    )
    for m in ix_pat.finditer(html):
        fact = {"concept": m.group(2), "contextRef": m.group(3), "value": _clean_xbrl_value(m.group(6))}
        if m.group(4):
            fact["decimals"] = m.group(4)
        if m.group(5):
            fact["unitRef"] = m.group(5)
        facts.append(fact)

    # Standalone XML style: <ffd:Foo contextRef="...">value</ffd:Foo>
    if not facts:
        xml_pat = re.compile(
            r'<ffd:([A-Za-z]+)[^>]*?contextRef="([^"]*)"[^>]*?(?:decimals="([^"]*)")?'
            r'[^>]*?(?:unitRef="([^"]*)")?[^>]*?>([\s\S]*?)</ffd:\1>',
            re.I,
        )
        for m in xml_pat.finditer(html):
            fact = {"concept": m.group(1), "contextRef": m.group(2), "value": _clean_xbrl_value(m.group(5))}
            if m.group(3):
                fact["decimals"] = m.group(3)
            if m.group(4):
                fact["unitRef"] = m.group(4)
            facts.append(fact)

    return facts


def _find_summary_context(by_ctx: dict) -> Optional[str]:
    summary_keys = {"TtlOfferingAmt", "NetFeeAmt", "TtlFeeAmt", "SubmissnTp", "FeeExhibitTp"}
    best, best_score = None, 0
    for ctx, facts_map in by_ctx.items():
        score = sum(1 for k in summary_keys if k in facts_map)
        if score > best_score:
            best, best_score = ctx, score
    return best


def _find_row_contexts(by_ctx: dict) -> list:
    row_keys = {"OfferingSctyTp", "SctyTitleTp", "FeeAmt", "MaxAggrgteOfferingAmt"}
    candidates = []
    for ctx, facts_map in by_ctx.items():
        if any(k in facts_map for k in row_keys):
            candidates.append(ctx)
    return sorted(candidates, key=lambda c: int(re.sub(r"\D", "", c) or "0"))


def _is_fee_exhibit(ex: dict) -> bool:
    """Validate that a parsed exhibit looks like an actual fee table,
    not an unrelated document (e.g. Exhibit 10.7) that the HTML-table
    fallback happened to extract rows from."""
    if ex.get("total_offering") and ex["total_offering"] > 0:
        return True
    if ex.get("total_fee") and ex["total_fee"] > 0:
        return True
    if ex.get("net_fee_due") and ex["net_fee_due"] > 0:
        return True
    for li in ex.get("line_items", []):
        if li.get("fee_amount") and li["fee_amount"] > 0:
            return True
        if li.get("max_aggregate_offering") and li["max_aggregate_offering"] > 0:
            return True
    return False


def parse_fee_exhibit(html: str, meta: dict) -> Optional[dict]:
    """Parse an Exhibit 107 HTML document into a structured dict."""
    facts = _extract_xbrl_facts(html)
    if not facts:
        return None

    by_ctx: dict = {}
    for f in facts:
        ctx = f["contextRef"]
        by_ctx.setdefault(ctx, {})[f["concept"]] = f["value"]

    summary_ctx = _find_summary_context(by_ctx)
    summary = by_ctx.get(summary_ctx, {}) if summary_ctx else {}
    row_ctxs = _find_row_contexts(by_ctx)

    line_items = []
    for i, ctx in enumerate(row_ctxs):
        c = by_ctx.get(ctx, {})
        item = {
            "row": i + 1,
            "security_type": _parse_security_type(c.get("OfferingSctyTp", "")),
            "class_title": c.get("SctyTitleTp", f"Security class {i+1}"),
            "fee_calc_rule": c.get("FeeCalcRuleNm"),
            "amount_registered": _parse_number(c.get("AmtSctiesRgstrd")),
            "price_per_unit": _parse_number(c.get("PricPerScty")),
            "max_aggregate_offering": _parse_number(c.get("MaxAggrgteOfferingAmt")),
            "fee_rate": _parse_number(c.get("FeeRate") or summary.get("FeeRate")),
            "fee_amount": _parse_number(c.get("FeeAmt")),
            "previously_paid": (c.get("PrevslyPdFlg", "false")).lower() == "true",
            "xbrl_context": ctx,
        }
        line_items.append(item)

    total_offering = (
        _parse_number(summary.get("TtlOfferingAmt"))
        or sum(r["max_aggregate_offering"] or 0 for r in line_items) or None
    )
    total_fee = (
        _parse_number(summary.get("TtlFeeAmt"))
        or sum(r["fee_amount"] or 0 for r in line_items) or None
    )
    net_fee_due = _parse_number(summary.get("NetFeeAmt")) or total_fee

    return {
        "cik": meta.get("cik", ""),
        "entity_name": meta.get("entity_name", ""),
        "accession_number": meta.get("accession_number", ""),
        "form_type": meta.get("form_type", ""),
        "filing_date": meta.get("filing_date", ""),
        "exhibit_url": meta.get("exhibit_url", ""),
        "exhibit_type": summary.get("FeeExhibitTp", "EX-FILING FEES"),
        "submission_type": summary.get("SubmissnTp", meta.get("form_type", "")),
        "line_items": line_items,
        "total_offering": total_offering,
        "total_fee": total_fee,
        "total_previously_paid": _parse_number(summary.get("TtlPrevlyPdAmt")) or 0,
        "total_fee_offset": _parse_number(summary.get("TtlFeeOffsetAmt")) or 0,
        "net_fee_due": net_fee_due,
        "fee_rate": (
            _parse_number(summary.get("FeeRate"))
            or next((r["fee_rate"] for r in line_items if r["fee_rate"]), None)
        ),
    }


def _try_parse_exhibit(cik: str, accession: str, filename: str, meta: dict) -> Optional[dict]:
    """Attempt to fetch and parse a single document as a fee exhibit.
    Returns None on failure instead of raising."""
    try:
        html = get_filing_document_text(cik, accession, filename)
        if not html:
            return None
        full_meta = {
            **meta,
            "cik": cik,
            "accession_number": accession,
            "exhibit_url": build_edgar_url(cik, accession, filename),
        }
        return parse_fee_exhibit(html, full_meta)
    except Exception:
        return None


def find_and_parse_fee_exhibit(cik: str, accession: str, meta: dict) -> Optional[dict]:
    """Locate the Exhibit 107 in a filing's document index and parse it.

    Uses a three-phase approach:
    1. Exact type match from SGML index-headers (e.g. "EX-FILING FEES")
    2. High-confidence filename patterns (e.g. "exfilingfee", "filingfee")
    3. Ambiguous patterns (e.g. "exhibit107") with post-parse validation
    """
    docs = get_filing_documents(cik, accession)

    # Phase 1: exact type match (reliable when index-headers are available)
    for d in docs:
        if (d.get("type") or "").upper() in _FEE_EXHIBIT_TYPES:
            result = _try_parse_exhibit(cik, accession, d["name"], meta)
            if result:
                return result

    # Phase 2: high-confidence filename patterns
    for d in docs:
        if any(p.search(d.get("name", "")) for p in _FEE_FILENAME_PRIMARY):
            result = _try_parse_exhibit(cik, accession, d["name"], meta)
            if result:
                return result

    # Phase 3: ambiguous patterns — try each candidate and validate
    for d in docs:
        if any(p.search(d.get("name", "")) for p in _FEE_FILENAME_SECONDARY):
            result = _try_parse_exhibit(cik, accession, d["name"], meta)
            if result and _is_fee_exhibit(result):
                return result

    return None

def search_fee_filings(
    company: str,
    form_types: Optional[list] = None,
    limit: int = 10,
) -> pd.DataFrame:
    """Find registration filings for a company (MCP: search_fee_filings).

    Returns a DataFrame of registration statements that may contain Exhibit 107.
    """
    resolved = resolve_cik(company)
    filings = get_company_filings(resolved["cik"])
    recent = filings["filings"]["recent"]

    form_filter = set(f.upper() for f in (form_types or REGISTRATION_FORMS))
    results = []

    for i in range(len(recent["accessionNumber"])):
        if len(results) >= limit:
            break
        form = recent["form"][i] or ""
        if form.upper() not in form_filter:
            continue
        acc = recent["accessionNumber"][i] or ""
        results.append({
            "company_name": resolved["name"],
            "form_type": form,
            "filing_date": recent["filingDate"][i] or "",
            "edgar_url": build_edgar_url(resolved["cik"], acc),
            "primary_document": recent["primaryDocument"][i] or ""
        })

    # Sort results by 'filing_date' in descending order (most recent to oldest)
    results.sort(key=lambda x: x.get('filing_date', ''), reverse=True)

    df = pd.DataFrame(results)
    print(f"{filings['name']} (CIK {resolved['cik']}) — {len(results)} registration filing(s)")
    return df


def get_fee_exhibit(
    accession_number: str,
    cik: str,
    exhibit_filename: Optional[str] = None,
) -> pd.DataFrame:
    """Fetch and parse a specific Exhibit 107 (MCP: get_fee_exhibit).

    Returns two DataFrames: summary (1 row) and line_items.
    """
    cik_padded = cik.zfill(10)
    filings = get_company_filings(cik_padded)
    meta = {"entity_name": filings.get("name", cik), "form_type": "", "filing_date": ""}

    recent = filings["filings"]["recent"]
    acc_normalized = accession_number.replace("/", "-")
    for i in range(len(recent["accessionNumber"])):
        if recent["accessionNumber"][i] == acc_normalized:
            meta["form_type"] = recent["form"][i] or ""
            meta["filing_date"] = recent["filingDate"][i] or ""
            break

    if exhibit_filename:
        html = get_filing_document_text(cik_padded, acc_normalized, exhibit_filename)
        if not html:
            raise ValueError(f"Could not fetch {exhibit_filename}")
        full_meta = {**meta, "cik": cik_padded, "accession_number": acc_normalized,
                     "exhibit_url": build_edgar_url(cik_padded, acc_normalized, exhibit_filename)}
        exhibit = parse_fee_exhibit(html, full_meta)
    else:
        exhibit = find_and_parse_fee_exhibit(cik_padded, acc_normalized, meta)

    if not exhibit:
        print(f"No Exhibit 107 found in filing {accession_number}")
        return pd.DataFrame()

    items_df = pd.DataFrame(exhibit["line_items"])
    summary = {k: v for k, v in exhibit.items() if k != "line_items"}
    print(f"{exhibit['entity_name']} — {exhibit['form_type']} ({exhibit['filing_date']})")
    print(f"  Total Offering: ${exhibit['total_offering']:, .2f}" if exhibit["total_offering"] else "  Total Offering: —")
    print(f"  Net Fee Due:    ${exhibit['net_fee_due']:, .2f}" if exhibit["net_fee_due"] else "  Net Fee Due: —")
    print(f"  Fee Rate:       {exhibit['fee_rate']*100:.6f}%" if exhibit["fee_rate"] else "  Fee Rate: —")
    print(f"  Line items: {len(exhibit['line_items'])}")
    return items_df


def get_company_fee_history(
    company: str,
    form_types: Optional[list] = None,
    limit: int = 5,
    start_date: Optional[str] = None,
) -> pd.DataFrame:
    """Fetch all fee exhibits for a company (MCP: get_company_fee_history).

    Returns a DataFrame with one row per filing's summary totals.
    """
    resolved = resolve_cik(company)
    filings = get_company_filings(resolved["cik"])
    recent = filings["filings"]["recent"]

    form_filter = set(f.upper() for f in (form_types or REGISTRATION_FORMS))
    start_dt = datetime.strptime(start_date, "%Y-%m-%d") if start_date else None

    matching = []
    for i in range(len(recent["accessionNumber"])):
        if len(matching) >= limit:
            break
        form = recent["form"][i] or ""
        date = recent["filingDate"][i] or ""
        if form.upper() not in form_filter:
            continue
        if start_dt and datetime.strptime(date, "%Y-%m-%d") < start_dt:
            continue
        matching.append({
            "acc": (recent["accessionNumber"][i] or "").replace("/", "-"),
            "form": form,
            "date": date,
        })

    exhibits = []
    errors = []
    for m in matching:
        try:
            ex = find_and_parse_fee_exhibit(
                resolved["cik"], m["acc"],
                {"entity_name": filings["name"], "form_type": m["form"], "filing_date": m["date"]},
            )
            if ex:
                exhibits.append(ex)
        except Exception as e:
            errors.append({"accession": m["acc"], "form": m["form"], "error": str(e)})

    if not exhibits and errors:
        print(f"  ☢ {len(matching)} filings found but all exhibit parses failed:")
        for err in errors:
            print(f"    {err['accession']} ({err['form']}): {err['error']}")

    rows = []
    for ex in exhibits:
        rows.append({
            "accession_number": ex["accession_number"],
            "form_type": ex["form_type"],
            "filing_date": ex["filing_date"],
            "entity_name": ex["entity_name"],
            "total_offering": ex["total_offering"],
            "total_fee": ex["total_fee"],
            "net_fee_due": ex["net_fee_due"],
            "fee_rate": ex["fee_rate"],
            "security_classes": len(ex["line_items"]),
            "exhibit_url": ex["exhibit_url"],
        })

    df = pd.DataFrame(rows)
    grand_offering = sum(r["total_offering"] or 0 for r in rows)
    grand_fees = sum(r["net_fee_due"] or 0 for r in rows)
    print(f"{filings['name']} — {len(exhibits)} exhibit(s) parsed")
    print(f"  Grand total offering: ${grand_offering:,.2f}")
    print(f"  Grand total net fees: ${grand_fees:,.2f}")
    return df


def all_fee_history(
    tickers: Optional[list] = None,
    form_types: Optional[list] = None,
    limit: int = 10,
    start_date: Optional[str] = None,
) -> pd.DataFrame:
    """Aggregate fee history across multiple tickers into one DataFrame.

    Calls get_company_fee_history() for each ticker and concatenates the
    results, adding a 'ticker' column to identify each company.
    Uses the notebook-level TICKERS and START_DATE when not specified.
    """
    tickers = tickers if tickers is not None else TICKERS
    start_date = start_date if start_date is not None else START_DATE

    frames = []
    for ticker in tickers:
        print(f"\n{'='*60}")
        try:
            df = get_company_fee_history(
                ticker, form_types=form_types, limit=limit, start_date=start_date or None,
            )
            if not df.empty:
                df.insert(0, "ticker", ticker)
                frames.append(df)
        except Exception as e:
            print(f"  Error for {ticker}: {e}")

    if not frames:
        print("\nNo fee history data found for any ticker.")
        return pd.DataFrame()

    combined = pd.concat(frames, ignore_index=True)
    print(f"\n{'='*60}")
    print(f"Combined: {len(combined)} exhibit(s) across {len(tickers)} entities")
    return combined


def compare_fee_rates(entities: list) -> pd.DataFrame:
    """Compare filing fee data across 2-8 entities (MCP: compare_fee_rates).

    Lists each parsed fee exhibit whose total_offering is greater than zero.
    """
    if len(entities) < 2 or len(entities) > 8:
        raise ValueError("Provide 2-8 company identifiers")

    rows = []
    for company in entities:
        try:
            resolved = resolve_cik(company)
            filings = get_company_filings(resolved["cik"])
            recent = filings["filings"]["recent"]
            form_filter = set(f.upper() for f in REGISTRATION_FORMS)

            for i in range(len(recent["accessionNumber"])):
                form = recent["form"][i] or ""
                if form.upper() not in form_filter:
                    continue
                acc = (recent["accessionNumber"][i] or "").replace("/", "-")
                date = recent["filingDate"][i] or ""
                ex = find_and_parse_fee_exhibit(
                    resolved["cik"], acc,
                    {"entity_name": filings["name"], "form_type": form, "filing_date": date},
                )
                if not ex or not ex.get("total_offering"):
                    continue

                rows.append({
                    "company": ex["entity_name"],
                    "ticker": resolved["ticker"],
                    "cik": resolved["cik"],
                    "form_type": ex["form_type"],
                    "filing_date": ex["filing_date"],
                    "accession_number": ex["accession_number"],
                    "total_offering": ex["total_offering"],
                    "net_fee_due": ex["net_fee_due"],
                    "fee_rate": ex["fee_rate"],
                })
        except Exception as e:
            rows.append({
                "company": company, "ticker": None, "cik": None,
                "form_type": None, "filing_date": None,
                "accession_number": None,
                "total_offering": None, "net_fee_due": None,
                "fee_rate": None, "error": str(e),
            })

    if not rows:
        print("No fee exhibits with total_offering > 0 found for the provided entities.")
    return pd.DataFrame(rows)


def get_ffd_concepts(
    company: str,
    concepts: Optional[list] = None,
    form_types: Optional[list] = None,
    limit: int = 10,
) -> pd.DataFrame:
    """Get all ffd: XBRL facts for a company (MCP: get_ffd_concepts_by_company).

    Merges two sources:
    - Phase 1: companyfacts API (fast, covers 424B/S-3/S-8)
    - Phase 2: Exhibit 107 parsing (backfills S-1/F-1 gap)
    """
    resolved = resolve_cik(company)
    concept_filter = set(concepts) if concepts else None
    form_filter = set(f.upper() for f in form_types) if form_types else None

    entries = []
    covered_accessions = set()

    # Phase 1: companyfacts
    facts_data = get_company_facts(resolved["cik"])
    ffd = (facts_data or {}).get("facts", {}).get("ffd", {})

    for tag, concept_data in ffd.items():
        if concept_filter and tag not in concept_filter:
            continue
        for unit, unit_entries in concept_data.get("units", {}).items():
            for entry in unit_entries:
                if form_filter and entry["form"].upper() not in form_filter:
                    continue
                accn = entry["accn"].replace("/", "-").strip()
                covered_accessions.add(accn);
                row = {
                    "concept": tag,
                    "label": concept_data.get("label", tag),
                    "unit": unit,
                    "value": entry["val"],
                    "filed": entry["filed"],
                    "form": entry["form"],
                    "accession": entry["accn"],
                    "period_end": entry["end"],
                    "source": "companyfacts",
                }
                if "fy" in entry:
                    row["fiscal_year"] = entry["fy"]
                if "fp" in entry:
                    row["fiscal_period"] = entry["fp"]
                if "start" in entry:
                    row["period_start"] = entry["start"]
                entries.append(row)

    # Phase 2: Exhibit 107 backfill
    filings = get_company_filings(resolved["cik"])
    recent = filings["filings"]["recent"]
    reg_forms = set(f.upper() for f in REGISTRATION_FORMS)
    backfill_attempted = 0

    for i in range(len(recent["accessionNumber"])):
        if backfill_attempted >= limit:
            break
        form = recent["form"][i] or ""
        if form.upper() not in reg_forms:
            continue
        if form_filter and form.upper() not in form_filter:
            continue
        acc = (recent["accessionNumber"][i] or "").replace("/", "-")
        if acc in covered_accessions:
            continue

        backfill_attempted += 1
        try:
            ex = find_and_parse_fee_exhibit(
                resolved["cik"], acc,
                {"entity_name": filings["name"], "form_type": form, "filing_date": recent["filingDate"][i] or ""},
            )
            if not ex:
                continue
            # Extract summary-level ffd concepts from parsed exhibit
            summary_facts = [
                ("TtlOfferingAmt", "Total Offering Amount", ex["total_offering"], "USD"),
                ("TtlFeeAmt", "Total Fee Amount", ex["total_fee"], "USD"),
                ("NetFeeAmt", "Net Fee Due", ex["net_fee_due"], "USD"),
                ("TtlPrevlyPdAmt", "Total Previously Paid", ex["total_previously_paid"], "USD"),
                ("TtlOffsetAmt", "Total Fee Offset", ex["total_fee_offset"], "USD"),
                ("FeeRate", "Fee Rate", ex["fee_rate"], "pure"),
            ]
            for concept_tag, label, value, u in summary_facts:
                if value is None:
                    continue
                if concept_filter and concept_tag not in concept_filter:
                    continue
                entries.append({
                    "concept": concept_tag, "label": label, "unit": u, "value": value,
                    "filed": ex["filing_date"], "form": ex["form_type"],
                    "accession": ex["accession_number"], "period_end": ex["filing_date"],
                    "source": "exhibit107",
                })
            # Per-line-item FeeAmt
            for item in ex["line_items"]:
                if item["fee_amount"] is None:
                    continue
                if concept_filter and "FeeAmt" not in concept_filter:
                    continue
                entries.append({
                    "concept": "FeeAmt", "label": f"Fee Amount — {item['class_title']}",
                    "unit": "USD", "value": item["fee_amount"],
                    "filed": ex["filing_date"], "form": ex["form_type"],
                    "accession": ex["accession_number"], "period_end": ex["filing_date"],
                    "source": "exhibit107",
                })
        except Exception:
            pass

    entries.sort(key=lambda e: e["filed"], reverse=True)
    df = pd.DataFrame(entries)
    entity_name = (facts_data or {}).get("entityName", resolved["name"])
    ex_count = sum(1 for e in entries if e["source"] == "exhibit107")
    print(f"{entity_name} — {len(entries)} ffd fact(s) including {ex_count} exhibit107 entries")
    return df

def _extract_text_context(text: str, query: str, window_size: int = 10) -> list[str]:
    """Extracts text snippets (window_size words before and after) for each query occurrence."""
    if not text or not query:
        return []

    # Use BeautifulSoup to strip HTML tags and get plain text
    soup = BeautifulSoup(text, 'html.parser')
    plain_text = soup.get_text()

    # Normalize whitespace and split into words
    words = re.findall(r'\b\w+\b', plain_text.lower())
    query_words = re.findall(r'\b\w+\b', query.lower())
    query_len = len(query_words)
    snippets = []

    if not query_words:
        return []

    for i in range(len(words) - query_len + 1):
        if words[i:i + query_len] == query_words:
            start_index = max(0, i - window_size)
            end_index = min(len(words), i + query_len + window_size)
            snippet_words = words[start_index:end_index]

            # Highlight the query in the snippet (optional, but good for readability)
            highlighted_snippet = []
            for j, word in enumerate(snippet_words):
                if j >= (i - start_index) and j < (i - start_index + query_len):
                    highlighted_snippet.append(f"**{word}**")
                else:
                    highlighted_snippet.append(word)

            snippets.append(" ".join(highlighted_snippet))

    return snippets

def search_filings_by_fee(
    query: str,
    forms: Optional[list] = None,
    start_date: Optional[str] = None,
    end_date: Optional[str] = None,
    limit: int = 20,
    tickers: Optional[list] = None,
) -> pd.DataFrame:
    """Full-text search scoped to registration statements (MCP: search_filings_by_fee).

    Searches EDGAR's EFTS index for the query, limited to registration form types.
    If tickers are provided or the notebook-level TICKERS is set, only results for those
    entities are returned.
    This modified version also performs a full-text search within primary documents
    and returns snippets of text around each occurrence.
    """
    search_forms = forms if forms else REGISTRATION_FORMS
    tickers = tickers if tickers is not None else TICKERS
    resolved_ciks = None
    if tickers:
        resolved_ciks = {resolve_cik(ticker)["cik"] for ticker in tickers}

    qs = f"q={requests.utils.quote(query)}&forms={','.join(search_forms)}"
    if start_date or end_date:
        qs += "&dateRange=custom"
        if start_date:
            qs += f"&startdt={start_date}"
        if end_date:
            qs += f"&enddt={end_date}"
    url = f"{_EFTS}/LATEST/search-index?{qs}&from=0&size={limit*2}" # Fetch more to allow for full-text filtering

    data = _sec_json(url)
    if not data:
        print("No results")
        return pd.DataFrame()

    hits = data.get("hits", {}).get("hits", [])
    total_initial_hits = data.get("hits", {}).get("total", {}).get("value", 0)

    all_results = []
    processed_count = 0
    for h in hits:
        if processed_count >= limit:
            break

        src = h.get("_source", {})
        result_ciks = [(cik or "").zfill(10) for cik in (src.get("ciks") or [])]
        if resolved_ciks and not any(cik in resolved_ciks for cik in result_ciks):
            continue

        company_name = (src.get("display_names") or [""])[0]
        company_name = re.sub(r"\s*\(CIK\s*\d+\)", "", company_name)
        company_name = re.sub(r"\s*\([A-Z0-9,\s.-]+\)\s*$", "", company_name).strip()

        cik = result_ciks[0] if result_ciks else ""
        accession_number = src.get("adsh") or h.get("_id", "").split(":")[0]

        # Get all documents for the accession number
        filing_docs = get_filing_documents(cik, accession_number)
        found_snippets_in_filing = False

        for doc in filing_docs:
            filename = doc.get("name")
            if filename and (filename.lower().endswith(('.htm', '.html'))):
                doc_text_html = get_filing_document_text(cik, accession_number, filename)
                if doc_text_html:
                    doc_text = BeautifulSoup(doc_text_html, "html.parser").get_text(separator=" ", strip=True)
                    snippets = _extract_text_context(doc_text, query)
                    if snippets:
                        for snippet in snippets:
                            all_results.append({
                                "company_name": company_name,
                                "accession_number": accession_number,
                                "filing_date": src.get("file_date", ""),
                                "form": src.get("form"),
                                "snippet": snippet,
                                "edgar_url": build_edgar_url(cik, accession_number, filename),
                            })
                        found_snippets_in_filing = True

        if found_snippets_in_filing:
            processed_count += 1 # Count only filings that yielded at least one snippet
        elif not filing_docs:
             print(f"Skipping {accession_number}: No documents found for this filing.")
        elif not cik or not accession_number:
             print(f"Skipping {accession_number}: Missing CIK or accession number.")

    df = pd.DataFrame(all_results)

    if resolved_ciks:
        print(f'Searching {len(resolved_ciks)} tickers, "{query}" — {len(df)} matching snippets from {processed_count} filing(s)')
    else:
        print(f'"{query}" — {total_initial_hits} initial hits, {len(df)} matching snippets from {processed_count} filing(s)')
    return df
def export_fee_data(
    company: str,
    mode: str = "fee_history",
    output_path: Optional[str] = None,
    limit: int = 10,
    form_types: Optional[list] = None,
) -> str:
    """Export fee data to CSV (MCP: export_fee_data).

    Modes: "fee_history" | "line_items" | "ffd_concepts"
    Returns the output file path.
    """
    resolved = resolve_cik(company)
    label = resolved["ticker"] or resolved["cik"]
    path = output_path or f"{label}_{mode}.csv"

    if mode == "ffd_concepts":
        df = get_ffd_concepts(company, form_types=form_types, limit=limit)
    elif mode == "line_items":
        df = _export_line_items(resolved["cik"], form_types, limit)
    else:
        df = get_company_fee_history(company, form_types=form_types, limit=limit)

    df.to_csv(path, index=False)
    print(f"Exported {len(df)} rows to {path}")
    return path


def _export_line_items(cik: str, form_types, limit) -> pd.DataFrame:
    """Helper: one row per security class across all filings."""
    filings = get_company_filings(cik)
    recent = filings["filings"]["recent"]
    form_filter = set(f.upper() for f in (form_types or REGISTRATION_FORMS))

    matching = []
    for i in range(len(recent["accessionNumber"])):
        if len(matching) >= limit:
            break
        form = recent["form"][i] or ""
        if form.upper() not in form_filter:
            continue
        matching.append({
            "acc": (recent["accessionNumber"][i] or "").replace("/", "-"),
            "form": form,
            "date": recent["filingDate"][i] or "",
        })

    rows = []
    for m in matching:
        try:
            ex = find_and_parse_fee_exhibit(
                cik, m["acc"],
                {"entity_name": filings["name"], "form_type": m["form"], "filing_date": m["date"]},
            )
            if not ex:
                continue
            for item in ex["line_items"]:
                rows.append({
                    "accession_number": ex["accession_number"],
                    "form_type": ex["form_type"],
                    "filing_date": ex["filing_date"],
                    "entity_name": ex["entity_name"],
                    "row_num": item["row"],
                    "security_type": item["security_type"],
                    "class_title": item["class_title"],
                    "fee_calc_rule": item["fee_calc_rule"],
                    "amount_registered": item["amount_registered"],
                    "price_per_unit": item["price_per_unit"],
                    "max_aggregate_offering": item["max_aggregate_offering"],
                    "fee_rate": item["fee_rate"],
                    "fee_amount": item["fee_amount"],
                    "previously_paid": item["previously_paid"],
                })
        except Exception:
            pass

    return pd.DataFrame(rows)
print("All tool functions loaded. \n\n User-Agent:", USER_AGENT, "is set - run the next cell the query the SEC API.")

In [ ]:
# @title Run this cell to gather registration filings for each entity.
if not TICKERS:
    print("No entities were found. Set ENTITIES above and rerun the input cell.")

all_search_filings = []
for ticker in TICKERS:
    print(f"\n{'='*60}")
    df = search_fee_filings(ticker)
    if not df.empty:
        all_search_filings.append(df)

if all_search_filings:
    combined_search_filings = pd.concat(all_search_filings, ignore_index=True)
    print(f"\n{'='*60}")
    print(f"Combined search filings: {len(combined_search_filings)} entries")
    display(combined_search_filings)
else:
    print("No search filings data found for any ticker.")

all_ffd = []
for ticker in TICKERS:
    print(f"\n{'='*60}")
    df = get_ffd_concepts(ticker)
    if not df.empty:
        df["ticker"] = ticker
        all_ffd.append(df)

if all_ffd:
    combined_ffd = pd.concat(all_ffd, ignore_index=True)
    print(f"\n{'='*60}")
    print(f"Combined FFD facts: {len(combined_ffd)} entries")
    display(combined_ffd)

if len(TICKERS) >= 2:
    comparison_df = compare_fee_rates(TICKERS)
    display(comparison_df)
    if not comparison_df.empty:
        totals = comparison_df.groupby("company", dropna=False)[["total_offering", "net_fee_due"]].sum(min_count=1)
        print("\nEntity totals:")
        for company, row in totals.iterrows():
            print(
                f"  {company} total offering=${row['total_offering']:,.2f}, "
                f"net fee due=${row['net_fee_due']:,.2f}"
            )
        grand = totals.sum()
        print(
            f"Grand totals:\n"
            f"  total offering: ${grand['total_offering']:,.2f}, \n"
            f"  net fees: ${grand['net_fee_due']:,.2f}\n"
        )
    else:
        print("No comparison data returned.")
else:
    print("Need 2+ tickers to compare. Add more to the TICKERS list above.")

search_df = pd.DataFrame()
if TEXT:
    print("Processing full-text search for ", TEXT, "— this may take a moment...")
    search_df = search_filings_by_fee(TEXT, start_date=START_DATE, end_date=END_DATE)
    display(search_df)
else:
    print("TEXT is empty; full-text search was skipped.")

In [ ]:
# @title Export corresponding report, concept and fact data, as well as any search results to spreadsheet.
EXPORT_MODES = ["fee_history", "line_items", "ffd_concepts"]

# Collect DataFrames per (ticker, mode) — no disk writes yet
sheets: dict[str, pd.DataFrame] = {}

for ticker in TICKERS:
    for mode in EXPORT_MODES:
        print(f"\n{'='*60}")
        print(f"Fetching {ticker} / {mode}...")
        try:
            if mode == "ffd_concepts":
                df = get_ffd_concepts(ticker, limit=10)
            elif mode == "line_items":
                resolved = resolve_cik(ticker)
                df = _export_line_items(resolved["cik"], None, 10)
            else:
                df = get_company_fee_history(ticker, limit=10)

            if not df.empty:
                df.insert(0, "ticker", ticker)
                sheet_name = f"{ticker}_{mode}"[:31]  # Excel limits sheet names to 31 chars
                sheets[sheet_name] = df
                print(f"  {len(df)} rows")
            else:
                print(f"  (no data)")
        except Exception as e:
            print(f"  Error: {e}")

# Include full-text search results if the search cell has been run
try:
    if "search_df" in vars() and not search_df.empty:
        query_label = re.sub(r"[^\w]", "_", TEXT)[:20]
        sheet_name = f"fts_{query_label}"[:31]
        sheets[sheet_name] = search_df
        print(f"\nAdding full-text search sheet '{sheet_name}': {len(search_df)} rows")
except Exception as e:
    print(f"  Could not add search results: {e}")

# Write all sheets into one .xlsx workbook
if sheets:
    today = date.today().isoformat()
    tickers_label = "_".join(TICKERS)[:40]
    xlsx_path = f"{tickers_label}_filing_fees_{today}.xlsx"

    with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
        for sheet_name, df in sheets.items():
            df.to_excel(writer, sheet_name=sheet_name, index=False)

    print(f"\n{'='*60}")
    print(f"Workbook saved: {xlsx_path}")
    print(f"  Sheets: {', '.join(sheets.keys())}")
    print(f"  Total rows: {sum(len(df) for df in sheets.values())}")
else:
    print("\nNo data to export.")